# Многократная временная валидация CatBoost

Здесь не меняются признаки и параметры модели. Цель ноутбука — измерить текущую версию на четырёх честных временных holdout и сохранить OOF-прогнозы как точку отсчёта для следующих экспериментов.

In [1]:
from __future__ import annotations

import gc
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import polars as pl

project_root = Path.cwd().resolve()
if not (project_root / 'src').is_dir():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import (
    FIRST_CATBOOST_SNAPSHOT_DIR,
    MULTIFOLD_VALIDATION_DIR,
    VALIDATION_ANCHORS,
    ensure_output_dirs,
)
from src.evaluation import feature_importance, save_json
from src.features import load_snapshots
from src.models import make_validation_model, predict_gmv
from src.validation import feature_columns, make_temporal_folds, rmsle

ensure_output_dirs()
print(f'Каталог срезов: {FIRST_CATBOOST_SNAPSHOT_DIR}')
print(f'Каталог результатов: {MULTIFOLD_VALIDATION_DIR}')

Каталог срезов: /Users/danasokol/Desktop/ML - соревы/OZON_GMV/artifacts/first_catboost_ltv/snapshots
Каталог результатов: /Users/danasokol/Desktop/ML - соревы/OZON_GMV/artifacts/multifold_validation


## 1. Загрузка готовых срезов и фолдов

Срезы не пересчитываются: используются те же признаки, которые уже были созданы первым baseline-ноутбуком.

In [2]:
snapshots = load_snapshots(FIRST_CATBOOST_SNAPSHOT_DIR, kind='train')
historical_anchors = sorted(snapshots)
reference_features = feature_columns(snapshots[historical_anchors[0]])

for anchor, snapshot in snapshots.items():
    assert feature_columns(snapshot) == reference_features, f'Разная схема признаков: {anchor}'

folds = make_temporal_folds(historical_anchors, VALIDATION_ANCHORS)
print(f'Число признаков: {len(reference_features)}')
for fold in folds:
    print(f'holdout={fold.validation_anchor}: train={list(fold.train_anchors)}')

Число признаков: 98
holdout=2025-10-22: train=[datetime.date(2025, 7, 2), datetime.date(2025, 7, 30), datetime.date(2025, 8, 27)]
holdout=2025-11-19: train=[datetime.date(2025, 7, 2), datetime.date(2025, 7, 30), datetime.date(2025, 8, 27), datetime.date(2025, 9, 24)]
holdout=2025-12-17: train=[datetime.date(2025, 7, 2), datetime.date(2025, 7, 30), datetime.date(2025, 8, 27), datetime.date(2025, 9, 24), datetime.date(2025, 10, 22)]
holdout=2026-01-14: train=[datetime.date(2025, 7, 2), datetime.date(2025, 7, 30), datetime.date(2025, 8, 27), datetime.date(2025, 9, 24), datetime.date(2025, 10, 22), datetime.date(2025, 11, 19)]


## 2. Обучение на каждом временном holdout

На каждой дате сравниваются наивный прогноз `GMV за предыдущие 30 дней` и неизменённый CatBoost. OOF означает, что прогноз для пользователя получен моделью, которая не видела его будущий GMV на этой дате.

In [3]:
metric_rows = []
oof_frames = []
importance_frames = []

for fold in folds:
    print(f'\nОбучение для holdout {fold.validation_anchor} ...')
    train = pl.concat([snapshots[anchor] for anchor in fold.train_anchors], how='vertical_relaxed')
    valid = snapshots[fold.validation_anchor]

    X_train = train.select(reference_features).to_pandas()
    y_train = np.log1p(train['target'].to_numpy())
    X_valid = valid.select(reference_features).to_pandas()
    y_valid = valid['target'].to_numpy()

    baseline_pred = valid['gmv_sum_30d'].to_numpy()
    model = make_validation_model()
    model.fit(
        X_train,
        y_train,
        eval_set=(X_valid, np.log1p(y_valid)),
        early_stopping_rounds=200,
        use_best_model=True,
    )
    catboost_pred = predict_gmv(model, X_valid)

    baseline_score = rmsle(y_valid, baseline_pred)
    catboost_score = rmsle(y_valid, catboost_pred)
    best_iteration = model.get_best_iteration()
    print(f'baseline RMSLE={baseline_score:.6f}; CatBoost RMSLE={catboost_score:.6f}; best_iteration={best_iteration}')

    metric_rows.append({
        'validation_anchor': fold.validation_anchor.isoformat(),
        'n_train_anchors': len(fold.train_anchors),
        'n_train_rows': train.height,
        'n_valid_rows': valid.height,
        'baseline_rmsle': baseline_score,
        'catboost_rmsle': catboost_score,
        'improvement': baseline_score - catboost_score,
        'best_iteration': best_iteration,
    })
    oof_frames.append(pd.DataFrame({
        'user_id': valid['user_id'].to_numpy(),
        'validation_anchor': fold.validation_anchor,
        'target': y_valid,
        'baseline_pred': baseline_pred,
        'catboost_pred': catboost_pred,
    }))
    importance_frames.append(
        feature_importance(model, reference_features).assign(
            validation_anchor=fold.validation_anchor.isoformat()
        )
    )

    del train, valid, X_train, y_train, X_valid, y_valid, model
    gc.collect()


Обучение для holdout 2025-10-22 ...
0:	learn: 2.2815918	test: 2.2957236	best: 2.2957236 (0)	total: 317ms	remaining: 9m 29s
200:	learn: 1.7076365	test: 1.7363544	best: 1.7363544 (200)	total: 55.7s	remaining: 7m 22s
400:	learn: 1.7002912	test: 1.7345305	best: 1.7345305 (400)	total: 1m 37s	remaining: 5m 40s
600:	learn: 1.6945009	test: 1.7342275	best: 1.7342110 (591)	total: 2m 13s	remaining: 4m 25s
800:	learn: 1.6892803	test: 1.7340167	best: 1.7339953 (788)	total: 2m 47s	remaining: 3m 28s
1000:	learn: 1.6844914	test: 1.7340590	best: 1.7339935 (891)	total: 3m 24s	remaining: 2m 42s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 1.733993458
bestIteration = 891

Shrink model to first 892 iterations.
baseline RMSLE=2.158052; CatBoost RMSLE=1.733993; best_iteration=891

Обучение для holdout 2025-11-19 ...
0:	learn: 2.2863558	test: 2.3382667	best: 2.3382667 (0)	total: 258ms	remaining: 7m 44s
200:	learn: 1.7121944	test: 1.7750851	best: 1.7750851 (200)	total: 44.6s	remaining: 5

## 3. Сводка и сохранение OOF

Будущие изменения признаков принимаются только по сравнению с этой таблицей и теми же OOF-фолдами.

In [4]:
metrics = pd.DataFrame(metric_rows).sort_values('validation_anchor')
oof = pd.concat(oof_frames, ignore_index=True)
importances = pd.concat(importance_frames, ignore_index=True)
mean_importance = (
    importances.groupby('feature', as_index=False)['importance']
    .mean()
    .sort_values('importance', ascending=False)
)

metrics.to_csv(MULTIFOLD_VALIDATION_DIR / 'metrics.csv', index=False)
oof.to_parquet(MULTIFOLD_VALIDATION_DIR / 'oof_predictions.parquet', index=False)
importances.to_csv(MULTIFOLD_VALIDATION_DIR / 'feature_importance_by_fold.csv', index=False)
mean_importance.to_csv(MULTIFOLD_VALIDATION_DIR / 'feature_importance_mean.csv', index=False)

summary = {
    'n_folds': len(metrics),
    'n_oof_rows': len(oof),
    'mean_baseline_rmsle': float(metrics['baseline_rmsle'].mean()),
    'mean_catboost_rmsle': float(metrics['catboost_rmsle'].mean()),
    'std_catboost_rmsle': float(metrics['catboost_rmsle'].std(ddof=0)),
    'global_oof_baseline_rmsle': rmsle(oof['target'].to_numpy(), oof['baseline_pred'].to_numpy()),
    'global_oof_catboost_rmsle': rmsle(oof['target'].to_numpy(), oof['catboost_pred'].to_numpy()),
}
save_json(summary, MULTIFOLD_VALIDATION_DIR / 'summary.json')

display(metrics)
print(summary)
display(mean_importance.head(25))

,validation_anchor,n_train_anchors,n_train_rows,n_valid_rows,baseline_rmsle,catboost_rmsle,improvement,best_iteration
0,2025-10-22,3,750000,250000,2.158052,1.733993,0.424059,891
1,2025-11-19,4,1000000,250000,2.194473,1.772562,0.421910,909
2,2025-12-17,5,1250000,250000,2.216610,1.770604,0.446006,878
3,2026-01-14,6,1500000,250000,2.195065,1.722584,0.472481,677


{'n_folds': 4, 'n_oof_rows': 1000000, 'mean_baseline_rmsle': 2.191049873828888, 'mean_catboost_rmsle': 1.7499358227703061, 'std_catboost_rmsle': 0.022030758736886324, 'global_oof_baseline_rmsle': 2.1911509037017822, 'global_oof_catboost_rmsle': 1.7500744950273721}


,feature,importance
97,to_ord_sum_90d,9.689036
88,to_ord_active_days_90d,9.626225
15,days_since_gmv,7.868063
44,observed_active_days_180d,7.543327
25,gmv_active_days_90d,6.695315
42,gmv_sum_90d,5.090755
19,days_since_to_ord,4.502294
34,gmv_mean_active_90d,2.551373
60,search_to_ord_sum_90d,2.246275
72,searches_sum_7d,2.153790
